# Workflow de Composition Musicale

**Module :** 04-Audio-Applications  
**Niveau :** Applications  
**Technologies :** MusicGen, Demucs, pydub, scipy, numpy  
**VRAM estimee :** ~14 GB  
**Duree estimee :** 60 minutes  

## Objectifs d'Apprentissage

- [ ] Generer de la musique a partir de descriptions textuelles avec MusicGen
- [ ] Separer les stems (voix, batterie, basse, autres) avec Demucs
- [ ] Remixer les stems (ajuster volumes, echanger des éléments)
- [ ] Appliquer des effets audio (reverb, EQ, compression)
- [ ] Créer des compositions multi-sections (intro, couplet, refrain)
- [ ] Exporter une production finale avec metadonnees

## Prerequis

- Notebooks Foundation (01-3 Basic Audio Opérations) et Advanced (02-3 MusicGen, 02-4 Demucs) completes
- GPU avec au moins 14 GB VRAM (RTX 3090 recommande)
- transformers (port MusicGen), demucs, pydub installes

**Navigation :** [Index](../README.md) | [<< Précédent](04-2-Transcription-Pipeline.ipynb) | [Suivant >>](04-4-Audio-Video-Sync.ipynb)

In [1]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration notebook
notebook_mode = "interactive"        # "interactive" ou "batch"
skip_widgets = False               # True pour mode batch MCP
debug_level = "INFO"

# Parametres MusicGen
musicgen_model = "facebook/musicgen-small"  # small, medium, large
generation_duration = 8            # Duree de generation en secondes
musicgen_device = "cuda"           # "cuda" ou "cpu"

# Parametres Demucs
demucs_model = "htdemucs"          # htdemucs, htdemucs_ft, mdx_extra

# Configuration pipeline
generate_audio = True
save_audio_files = True
apply_effects = True               # Appliquer les effets audio


In [2]:
# Parameters
musicgen_model = "facebook/musicgen-medium"
notebook_mode = "batch"
skip_widgets = True


L'environnement de travail est configure avec les paramètres Papermill. La cellule suivante charge les bibliotheques Python necessaires : generation audio, traitement de signal et utilitaires de composition.

In [3]:
# Setup environnement et imports
import os
import sys
import json
import time
import struct
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional, Tuple
from io import BytesIO
import logging

# Silencer le bruit HTTP de huggingface_hub AVANT tout import du Hub : chaque
# requete emet une ligne "INFO:httpx" en stderr, qui gonfle la sortie committée
# et fait rougir le ratchet Output-flood (cf. #23967). Deux leviers, deux sources :
# la variable couvre les logs du Hub, le niveau du logger couvre httpx lui-meme.
os.environ.setdefault("HF_HUB_VERBOSITY", "error")
logging.getLogger("httpx").setLevel(logging.WARNING)

import numpy as np
from IPython.display import Audio, display, HTML

# Resolution GENAI_ROOT
GENAI_ROOT = Path.cwd()
while GENAI_ROOT.name != 'GenAI' and len(GENAI_ROOT.parts) > 1:
    GENAI_ROOT = GENAI_ROOT.parent

HELPERS_PATH = GENAI_ROOT / 'shared' / 'helpers'
if HELPERS_PATH.exists():
    sys.path.insert(0, str(HELPERS_PATH.parent))
    try:
        from helpers.audio_helpers import (
            load_audio, save_audio, play_audio,
            plot_waveform, plot_spectrogram,
            display_audio_array, display_audio_file
        )
        print("Helpers audio importes")
    except ImportError as e:
        print(f"Helpers audio non disponibles - mode autonome : {e}")

# Repertoire de sortie
OUTPUT_DIR = GENAI_ROOT / 'outputs' / 'audio' / 'composition'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STEMS_DIR = OUTPUT_DIR / 'stems'
STEMS_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=getattr(logging, debug_level))
logger = logging.getLogger('music_composition')

print(f"Workflow de Composition Musicale")
print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode : {notebook_mode}")
print(f"MusicGen : {musicgen_model} | Demucs : {demucs_model}")
print(f"Sortie : {OUTPUT_DIR.relative_to(GENAI_ROOT)}")


Helpers audio importes
Workflow de Composition Musicale
Date : 2026-11-02 11:00:47
Mode : batch
MusicGen : facebook/musicgen-medium | Demucs : htdemucs
Sortie : outputs\audio\composition


Les modules Python sont importes. Cette cellule charge les cles API depuis le fichier `.env` en remontant l'arborescence de repertoires pour s'adapter aux différents contextes d'exécution.

In [4]:
# Chargement robuste de la configuration .env (Pattern A - Papermill-safe)
from dotenv import load_dotenv
import os
from pathlib import Path

current_path = Path.cwd()
genai_path = None

# Remonter jusqu'au dossier GenAI
while len(current_path.parts) > 1:
    if current_path.name == "GenAI":
        genai_path = current_path
        break
    current_path = current_path.parent

if genai_path:
    env_path = genai_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path.name}")

# Verification des dependances audio
import torch

# GPU check
if torch.cuda.is_available():
    print(f"GPU disponible : {torch.cuda.get_device_name(0)}")
    device = "cuda"
else:
    print("GPU non disponible - basculement CPU (lent)")
    device = "cpu"

# Verification explicite et bruyante des dependances (issue #10985 - PR #11385)
# Avant: try/except silencieux -> committait l'absence comme un succes apparent.
# Apres: RuntimeError honnete des qu'une dependance requise manque (generate_audio=True).
missing_pkgs = []

try:
    from transformers import MusicgenForConditionalGeneration, AutoProcessor
    musicgen_available = True
    print("MusicGen disponible via transformers")
except ImportError as e:
    musicgen_available = False
    missing_pkgs.append(("transformers (MusicGen)", "transformers", str(e)))

try:
    import demucs
    demucs_available = True
    print("demucs disponible")
except ImportError as e:
    demucs_available = False
    missing_pkgs.append(("demucs (separation stems)", "demucs", str(e)))

try:
    from pydub import AudioSegment
    from scipy.signal import butter, sosfilt
    print("pydub et scipy disponibles")
except ImportError as e:
    missing_pkgs.append(("pydub + scipy (remixage)", "pydub scipy", str(e)))

if missing_pkgs:
    print("")
    print("=" * 70)
    print("DEPENDANCES MANQUANTES (issue #10985 - verdict RECOVERABLE-LOCAL)")
    print("=" * 70)
    for label, pip_pkg, err in missing_pkgs:
        print(f"  - {label}: {err}")
    install_cmd = "pip install " + " ".join(sorted({p for _, p, _ in missing_pkgs}))
    print(f"")
    print(f"Installation : {install_cmd}")
    print(f"")
    if generate_audio:
        # Regle F : reparer, pas contourner. generate_audio=True exige les deps.
        raise RuntimeError(
            f"Dependances audio manquantes pour generate_audio=True. "
            f"Installez : {install_cmd}"
        )
    print("Mode degrade : generate_audio=False, les cellules en aval seront silencieuses (DESIGNE).")


GPU disponible : NVIDIA GeForce RTX 4060 Laptop GPU


MusicGen disponible via transformers
demucs disponible


pydub et scipy disponibles


### Interpretation : Configuration de l'environnement

| Composant | Statut | Utilite dans le pipeline |
|-----------|--------|--------------------------|
| GPU (RTX 4060 Laptop) | Detecte | Acceleration de MusicGen et Demucs |
| transformers (MusicGen) | Installe | Generation musicale operationnelle |
| demucs | Installe | Separation de stems operationnelle |
| pydub + scipy | Disponibles | Remixage et effets operationnels |

**Points cles** :
1. MusicGen et Demucs sont installes : la generation et la separation s'executent integralement
2. Le pipeline complet est operationnel (generation -> separation -> remix -> effets -> export)
3. pydub et scipy couvrent le remixage et les effets sur les stems generes

> **Note** : Pour une exécution complete, les dependances transformers (port MusicGen) et demucs sont requises (`pip install transformers demucs`).

## Section 1 : Generation musicale avec MusicGen

MusicGen genere de la musique a partir de descriptions textuelles. Le modèle produit des fichiers audio de qualite studio.

| Modèle | Paramètres | VRAM | Qualite | Duree max |
|--------|-----------|------|---------|----------|
| musicgen-small | 300M | ~4 GB | Correcte | 30s |
| musicgen-medium | 1.5B | ~8 GB | Bonne | 30s |
| musicgen-large | 3.3B | ~14 GB | Excellente | 30s |

Le prompt textuel guide le style, l'instrumentation et l'ambiance de la musique generee.

### Preparation de la generation

La generation musicale avec MusicGen commence par définir des descriptions textuelles (prompts) pour chaque section de la composition. Ces prompts doivent etre descriptifs et précis pour guider le modèle vers le résultat souhaite.

**Stratégies de prompting** :
- **Style** : Genre musical (electronic, orchestral, jazz, rock)
- **Instrumentation** : Instruments predominants (piano, strings, drums, synth)
- **Ambiance** : Atmosphere (calm, energetic, epic, melancholic)
- **Tempo implicite** : Le modèle deduit le tempo de la description

**Exemples de prompts** :
- "soft ambient piano with gentle strings" → Intro calme
- "upbeat electronic with synth pads and light drums" → Verse rythme
- "powerful orchestral with drums and brass" → Chorus epique

La cellule suivante charge le modèle MusicGen et genere les sections définies dans `track_descriptions`.

In [5]:
# Generation musicale avec MusicGen
print("GENERATION MUSICALE - MUSICGEN")
print("=" * 50)

generated_tracks = {}

# Descriptions pour differentes sections d'une composition
track_descriptions = {
    "intro": "soft ambient piano with gentle strings, calm and peaceful, cinematic intro",
    "verse": "upbeat electronic music with synth pads and light drums, energetic but not aggressive",
    "chorus": "powerful orchestral music with drums and brass, epic and uplifting, full arrangement"
}

if generate_audio and musicgen_available:
    print(f"Chargement du modele {musicgen_model}...")
    start_time = time.time()
    mg_processor = AutoProcessor.from_pretrained(musicgen_model)
    mg_model = MusicgenForConditionalGeneration.from_pretrained(
        musicgen_model, torch_dtype=torch.float16, device_map=device)
    mg_frame_rate = mg_model.config.audio_encoder.frame_rate
    mg_sample_rate = mg_model.config.audio_encoder.sampling_rate
    load_time = time.time() - start_time
    print(f"Modele charge en {load_time:.1f}s (frame rate {mg_frame_rate} Hz, SR {mg_sample_rate} Hz)")

    for section_name, description in track_descriptions.items():
        print(f"\n--- Generation : {section_name} ---")
        print(f"Prompt : {description}")

        start_time = time.time()
        inputs = mg_processor(text=[description], padding=True, return_tensors="pt").to(device)
        audio_values = mg_model.generate(
            **inputs, do_sample=True, guidance_scale=3.0,
            max_new_tokens=int(generation_duration * mg_frame_rate)
        )
        gen_time = time.time() - start_time

        # Extraire le numpy array (float32 : fp16 du modele non supporte par soundfile)
        audio_data = audio_values[0, 0].cpu().float().numpy()
        sample_rate = mg_sample_rate

        generated_tracks[section_name] = {
            "audio": audio_data,
            "sr": sample_rate,
            "duration": len(audio_data) / sample_rate,
            "time": gen_time,
            "description": description
        }

        print(f"Genere en {gen_time:.1f}s | Duree : {len(audio_data)/sample_rate:.1f}s | SR : {sample_rate}Hz")
        display_audio_array(audio_data, sample_rate)

        # Sauvegarder
        if save_audio_files:
            import soundfile as sf
            filepath = OUTPUT_DIR / f"musicgen_{section_name}.wav"
            sf.write(str(filepath), audio_data, sample_rate)
            print(f"Sauvegarde : {filepath.name}")

    # Recapitulatif
    print(f"\nRecapitulatif des generations :")
    print(f"{'Section':<12} {'Duree (s)':<12} {'Temps gen (s)':<15}")
    print("-" * 39)
    for name, data in generated_tracks.items():
        print(f"{name:<12} {data['duration']:<12.1f} {data['time']:<15.1f}")

    # Liberer la memoire GPU
    del mg_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"\nMemoire GPU liberee")
else:
    if not musicgen_available:
        print("MusicGen non disponible - installation requise : pip install transformers")
    else:
        print("Generation desactivee")


GENERATION MUSICALE - MUSICGEN
Chargement du modele facebook/musicgen-medium...


[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 2047), got 2048. This may result in unexpected behavior.


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 2047), got 2048. This may result in unexpected behavior.


Loading weights:   0%|          | 0/996 [00:00<?, ?it/s]

Modele charge en 21.6s (frame rate 50 Hz, SR 32000 Hz)

--- Generation : intro ---
Prompt : soft ambient piano with gentle strings, calm and peaceful, cinematic intro


Genere en 59.0s | Duree : 7.9s | SR : 32000Hz


... (4 outputs left out)

Sauvegarde : musicgen_verse.wav

--- Generation : chorus ---
Prompt : powerful orchestral music with drums and brass, epic and uplifting, full arrangement


Genere en 56.0s | Duree : 7.9s | SR : 32000Hz


<audio embedded (audio/wav, 508204 bytes)>

Sauvegarde : musicgen_chorus.wav

Recapitulatif des generations :
Section      Duree (s)    Temps gen (s)  
---------------------------------------
intro        7.9          59.0           
verse        7.9          54.3           
chorus       7.9          56.0           

Memoire GPU liberee


### Interpretation : Generation MusicGen

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| Temps de generation | *runtime machine-dep* : ~2-5s par section (small), ~38-42s en medium (ce run, RTX 4060 Laptop) | ~5x le temps réel sur GPU laptop, reste praticable pour prototyper |
| Qualite | 32kHz mono | Suffisante pour prototypage, pas pour production finale |
| Contrôle | Via le prompt textuel | Le style depend fortement de la description |

**Points cles** :
1. Des prompts detailles (instrumentation, ambiance, tempo) ameliorent la qualite
2. La generation est non-déterministe : chaque exécution produit un résultat différent
3. Le modèle `large` offre une qualite significativement superieure au `small`

## Section 2 : Separation de stems avec Demucs

Demucs separe un mix musical en 4 stems :

| Stem | Description | Utilisation remix |
|------|-------------|-------------------|
| `drums` | Batterie, percussions | Ajuster le rythme |
| `bass` | Ligne de basse | Modifier les fondations |
| `other` | Melodie, harmonie (synths, guitares) | Éléments melodiques |
| `vocals` | Voix (si presentes) | Isoler ou supprimer la voix |

La separation permet de remixer chaque élément independamment.

In [6]:
# Separation de stems avec Demucs
print("SEPARATION DE STEMS - DEMUCS")
print("=" * 50)

separated_stems = {}

if generate_audio and demucs_available and generated_tracks:
    # Utiliser le track "chorus" (le plus riche)
    target_track = "chorus"
    if target_track not in generated_tracks:
        target_track = list(generated_tracks.keys())[0]

    track_data = generated_tracks[target_track]
    print(f"Separation du track : {target_track}")
    print(f"Duree : {track_data['duration']:.1f}s | SR : {track_data['sr']}Hz")

    # Sauvegarder le fichier pour Demucs
    import soundfile as sf
    input_path = OUTPUT_DIR / f"demucs_input_{target_track}.wav"
    sf.write(str(input_path), track_data['audio'], track_data['sr'])

    # Executer Demucs
    print(f"\nExecution de Demucs ({demucs_model})...")
    start_time = time.time()

    import subprocess
    result = subprocess.run(
        [sys.executable, "-m", "demucs", "--name", demucs_model,
         "--out", str(STEMS_DIR), str(input_path)],
        capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=300
    )

    sep_time = time.time() - start_time

    if result.returncode == 0:
        print(f"Separation terminee en {sep_time:.1f}s")

        # Charger les stems
        stems_path = STEMS_DIR / demucs_model / input_path.stem
        stem_names = ["drums", "bass", "other", "vocals"]

        print(f"\nStems separes :")
        for stem_name in stem_names:
            stem_file = stems_path / f"{stem_name}.wav"
            if stem_file.exists():
                stem_audio, stem_sr = sf.read(str(stem_file))
                # Si stereo, prendre la moyenne pour la lecture
                if len(stem_audio.shape) > 1:
                    stem_mono = stem_audio.mean(axis=1)
                else:
                    stem_mono = stem_audio

                energy = np.sqrt(np.mean(stem_mono ** 2))
                separated_stems[stem_name] = {
                    "audio": stem_audio,
                    "mono": stem_mono,
                    "sr": stem_sr,
                    "energy": energy,
                    "path": stem_file
                }

                print(f"  {stem_name:8s} | Energie RMS : {energy:.4f} | Shape : {stem_audio.shape}")
                display_audio_array(stem_mono, stem_sr)
            else:
                print(f"  {stem_name:8s} | Fichier non trouve")
    else:
        print(f"Erreur Demucs : {result.stderr[:200]}")

elif not demucs_available:
    print("Demucs non disponible - pip install demucs")
elif not generated_tracks:
    print("Pas de tracks generees - executez la Section 1 d'abord")
else:
    print("Separation desactivee")


SEPARATION DE STEMS - DEMUCS
Separation du track : chorus
Duree : 7.9s | SR : 32000Hz

Execution de Demucs (htdemucs)...


Separation terminee en 12.6s

Stems separes :
  drums    | Energie RMS : 0.0181 | Shape : (350154, 2)


<audio embedded (audio/wav, 700352 bytes)>

  bass     | Energie RMS : 0.1065 | Shape : (350154, 2)


<audio embedded (audio/wav, 700352 bytes)>

  other    | Energie RMS : 0.0814 | Shape : (350154, 2)


<audio embedded (audio/wav, 700352 bytes)>

  vocals   | Energie RMS : 0.0001 | Shape : (350154, 2)


<audio embedded (audio/wav, 700352 bytes)>

### Preparation de la separation

La separation de stems est une technologie qui permet d'isoler les différents éléments d'un mix audio. Demucs utilise des reseaux de neurones profonds pour cette tâche.

**Workflow de separation** :
1. **Sauvegarde** : Le fichier audio est ecrit sur disque (format WAV)
2. **Inference** : Demucs traite le fichier et genere 4 stems
3. **Chargement** : Les stems separes sont charges en memoire
4. **Analyse** : L'energie RMS de chaque stem est calculee

**Modèles Demucs** :
- **htdemucs** : Modèle par defaut, bon compromis qualite/vitesse
- **htdemucs_ft** : Version fine-tuned, meilleure qualite
- **mdx_extra** : Modèle MDX, specialise pour certains types de musique

La separation prend environ *runtime machine-dep* : quelques secondes à 30 secondes pour 8 secondes d'audio sur GPU, selon le modèle et le matériel.

### Interpretation : Separation de stems

| Stem | Energie RMS typique | Observation |
|------|-------------------|-------------|
| drums | Elevee | Batterie bien isolee si presente |
| bass | Moyenne | Basses frequences extraites |
| other | Variable | Contient melodie et harmonie |
| vocals | Faible (musique instrumentale) | Vide si pas de voix dans l'original |

> **Note technique** : Pour de la musique generee par MusicGen (instrumentale), le stem `vocals` sera quasi-vide. C'est normal.

***

## Exercice : Analyseur de mix audio

**Duree estimee :** 20-25 minutes

### Objectif
Créer une fonction qui analyse un fichier audio et identifie les composants frequentiels (basses, mediums, aigus) en calculant l'energie de chaque bande.

### Contexte
Avant de remixer un morceau, il est essentiel d'analyser son contenu frequentiel pour comprendre quels éléments dominent. Cette analyse permet de prendre des decisions de mixage eclairees (boost ou attenuation des bandes).

### Instructions
1. Charger un fichier audio avec pydub
2. Convertir en numpy array
3. Separer en 3 bandes (basses < 300Hz, mediums 300-4000Hz, aigus > 4000Hz)
4. Calculer l'energie RMS de chaque bande
5. Retourner un rapport structure avec recommandations de mixage

> **Indices :**
- `# Étape 1` : Charger l'audio avec `AudioSegment.from_file()` puis convertir en numpy
- `# Étape 2` : Utiliser `scipy.signal.butter()` et `scipy.signal.sosfilt()` pour les filtres
- `# Étape 3` : L'energie RMS = `np.sqrt(np.mean(audio ** 2))`
- `# Indice` : `butter(4, 300, btype='low', fs=sr, output='sos')` pour les basses
- `# Indice` : Comparer les energies pour recommander quelles bandes ajuster

In [7]:
# TODO: Implementer l'analyseur de mix audio
def analyze_frequency_bands(audio_path: str) -> dict:
    """
    Analyse les composants frequentiels d'un fichier audio.
    
    Args:
        audio_path: Chemin vers le fichier audio
    
    Returns:
        Dict avec energie_bass, energie_mid, energie_treble, recommandations
    """
    # Etape 1 : Charger l'audio
    # Indice: AudioSegment.from_file(audio_path) puis np.array(audio.get_array_of_samples())
    pass  # TODO etudiant
    
    # Etape 2 : Appliquer les filtres passe-bande
    # Indice: from scipy.signal import butter, sosfilt
    # Indice: bass = sosfilt(butter(4, 300, btype='low', fs=sr, output='sos'), samples)
    pass  # TODO etudiant
    
    # Etape 3 : Calculer les energies RMS
    # Indice: rms = np.sqrt(np.mean(band ** 2))
    pass  # TODO etudiant
    
    # Etape 4 : Generer des recommandations
    # Indice: Si bass_energy > mid_energy * 2 -> "Basses predominantes, reduire de -3dB"
    return {"status": "Exercice a completer"}

# Test avec un fichier audio
# report = analyze_frequency_bands("mon_audio.wav")
# print(f"Basses : {report['energie_bass']:.4f}")
# print(f"Mediums : {report['energie_mid']:.4f}")
# print(f"Aigus : {report['energie_treble']:.4f}")
# for reco in report['recommandations']:
#     print(f"  -> {reco}")
print("Exercice a completer")

Exercice a completer


## Section 3 : Remixage des stems

Le remixage consiste a recombiner les stems avec des niveaux différents pour créer une nouvelle version.

| Opération | Description | Paramètre |
|-----------|-------------|----------|
| Volume | Ajuster le gain de chaque stem | dB |
| Mute | Supprimer un stem | On/Off |
| Solo | Isoler un stem | On/Off |
| Pan | Position stereo | Gauche/Centre/Droite |

### Théorie du remixage

Le remixage consiste a equilibrer les niveaux relatifs des différentes pistes (stems) pour créer un mix coherent. C'est une étape critique de la production musicale.

**Concepts fondamentaux** :
- **Gain** : Amplification ou attenuation d'un signal (0.0 = mute, 1.0 = unite)
- **Balance** : Repartition de l'energie entre les stems
- **Clipping** : Distorsion lorsque le signal depasse la capacite maximale
- **Headroom** : Marge de securite avant le clipping (typiquement -0.5 a -3 dBFS)

**Stratégies de mix** :
- **Mix statique** : Gains constants tout au long du morceau
- **Mix dynamique** : Gains variables (automation) pour créer du mouvement
- **Mix par section** : Equilibrage différent pour intro, verse, chorus

La normalisation finale (peak a 0.95) garantit qu'aucun echantillon ne depasse la capacite maximale, evitant le clipping.

In [8]:
# Remixage des stems
print("REMIXAGE DES STEMS")
print("=" * 50)

def remix_stems(stems: Dict, mix_config: Dict[str, float],
                sample_rate: int) -> np.ndarray:
    """Remixe les stems avec les gains specifies.

    Args:
        stems: Dict {name: {"mono": np.array, ...}}
        mix_config: Dict {name: gain_linear} (0.0 = mute, 1.0 = normal)
        sample_rate: Taux d'echantillonnage

    Returns:
        np.ndarray: Mix final
    """
    # Determiner la longueur maximale
    max_len = max(len(s['mono']) for s in stems.values())

    mix = np.zeros(max_len, dtype=np.float32)

    for stem_name, stem_data in stems.items():
        gain = mix_config.get(stem_name, 1.0)
        mono = stem_data['mono']

        # Pad si necessaire
        if len(mono) < max_len:
            mono = np.pad(mono, (0, max_len - len(mono)))

        mix += mono * gain

    # Normalisation pour eviter le clipping
    peak = np.max(np.abs(mix))
    if peak > 0:
        mix = mix / peak * 0.95

    return mix

if separated_stems:
    sr = list(separated_stems.values())[0]['sr']

    # Differentes configurations de mix
    mix_configs = {
        "Original (equilibre)": {"drums": 1.0, "bass": 1.0, "other": 1.0, "vocals": 1.0},
        "Instrumental (sans voix)": {"drums": 1.0, "bass": 1.0, "other": 1.0, "vocals": 0.0},
        "Rythme only": {"drums": 1.2, "bass": 0.8, "other": 0.0, "vocals": 0.0},
        "Melodique (sans batterie)": {"drums": 0.0, "bass": 0.5, "other": 1.2, "vocals": 0.8},
    }

    remix_results = {}
    for mix_name, config in mix_configs.items():
        print(f"\n--- {mix_name} ---")
        config_str = " | ".join([f"{k}: {v:.1f}" for k, v in config.items()])
        print(f"  Config : {config_str}")

        mix = remix_stems(separated_stems, config, sr)
        remix_results[mix_name] = mix

        display_audio_array(mix, sr)

        if save_audio_files:
            import soundfile as sf
            safe_name = mix_name.lower().replace(' ', '_').replace('(', '').replace(')', '')
            filepath = OUTPUT_DIR / f"remix_{safe_name}.wav"
            sf.write(str(filepath), mix, sr)
            print(f"  Sauvegarde : {filepath.name}")

    print(f"\n{len(remix_results)} versions de remix creees")
else:
    print("Pas de stems disponibles - executez la Section 2 d'abord")
    remix_results = {}


REMIXAGE DES STEMS

--- Original (equilibre) ---
  Config : drums: 1.0 | bass: 1.0 | other: 1.0 | vocals: 1.0


<audio embedded (audio/wav, 700352 bytes)>

  Sauvegarde : remix_original_equilibre.wav

--- Instrumental (sans voix) ---
  Config : drums: 1.0 | bass: 1.0 | other: 1.0 | vocals: 0.0


<audio embedded (audio/wav, 700352 bytes)>

  Sauvegarde : remix_instrumental_sans_voix.wav

--- Rythme only ---
  Config : drums: 1.2 | bass: 0.8 | other: 0.0 | vocals: 0.0


<audio embedded (audio/wav, 700352 bytes)>

  Sauvegarde : remix_rythme_only.wav

--- Melodique (sans batterie) ---
  Config : drums: 0.0 | bass: 0.5 | other: 1.2 | vocals: 0.8


<audio embedded (audio/wav, 700352 bytes)>

  Sauvegarde : remix_melodique_sans_batterie.wav

4 versions de remix creees


### Interpretation : Remixage

| Version | Drums | Bass | Other | Vocals | Caractère |
|---------|-------|------|-------|--------|----------|
| Original | 1.0 | 1.0 | 1.0 | 1.0 | Fidele au mix original |
| Instrumental | 1.0 | 1.0 | 1.0 | 0.0 | Karaoke / fond musical |
| Rythme only | 1.2 | 0.8 | 0.0 | 0.0 | Base rythmique |
| Melodique | 0.0 | 0.5 | 1.2 | 0.8 | Ambiance melodique |

**Points cles** :
1. La normalisation previent le clipping lors de l'amplification
2. Combiner separation et remixage ouvre des possibilites creatives infinies
3. Les gains >1.0 amplifient, <1.0 attenuent, 0.0 coupe completement

## Section 4 : Effets audio

Les effets audio transforment le caractère sonore d'un signal :

| Effet | Description | Paramètre principal |
|-------|-------------|-------------------|
| Reverb | Simulation d'espace (salle, cathedral) | Decay time |
| EQ | Egalisation des frequences | Bandes de frequences |
| Compression | Reduction de la dynamique | Ratio, threshold |
| Fade | Fondu d'entree/sortie | Duree (ms) |

### Théorie des effets audio

Les effets audio modifient les caracteriques sonores d'un signal. Comprendre leur fonctionnement est essentiel pour un mixage professionnel.

**Reverb (Reverberation)** :
Simule la reflexion du son dans un espace physique. Une reverb est composee de :
- **Pre-delay** : Temps avant le debut des reflexions (20-100ms)
- **Early reflections** : Premières reflexions directes
- **Tail** : Reverberation diffuse, decroissance exponentielle
- **Decay time** : Duree de la decroissance (0.3-3s)

**EQ (Egalisation)** :
Modifie le balance des frequences :
- **Basses** (<300 Hz) : Fondation, chaleur
- **Mediums** (300 Hz - 4 kHz) : Corps, presence
- **Aigus** (>4 kHz) : Clarte, brillance

La cellule suivante implemente ces effets de maniere simplifiee avec Python pur et scipy.

In [9]:
# Application d'effets audio
print("EFFETS AUDIO")
print("=" * 50)

def apply_reverb(audio: np.ndarray, sr: int, decay: float = 0.3,
                 delay_ms: float = 40) -> np.ndarray:
    """Applique une reverb simple (delay + feedback)."""
    delay_samples = int(sr * delay_ms / 1000)
    output = np.copy(audio).astype(np.float64)

    # Plusieurs taps de delay pour simuler les reflexions
    for tap_mult in [1.0, 1.7, 2.3, 3.1]:
        tap_delay = int(delay_samples * tap_mult)
        tap_gain = decay ** tap_mult
        if tap_delay < len(output):
            output[tap_delay:] += audio[:len(audio) - tap_delay] * tap_gain

    # Normalisation
    peak = np.max(np.abs(output))
    if peak > 0:
        output = output / peak * 0.95

    return output.astype(np.float32)

def apply_eq(audio: np.ndarray, sr: int,
             bass_gain: float = 1.0, mid_gain: float = 1.0,
             treble_gain: float = 1.0) -> np.ndarray:
    """EQ 3 bandes simple (basses, mediums, aigus)."""
    from scipy.signal import butter, sosfilt

    # Filtres passe-bande
    sos_low = butter(4, 300, btype='low', fs=sr, output='sos')
    sos_mid = butter(4, [300, 4000], btype='band', fs=sr, output='sos')
    sos_high = butter(4, 4000, btype='high', fs=sr, output='sos')

    bass = sosfilt(sos_low, audio) * bass_gain
    mid = sosfilt(sos_mid, audio) * mid_gain
    treble = sosfilt(sos_high, audio) * treble_gain

    result = (bass + mid + treble).astype(np.float32)

    peak = np.max(np.abs(result))
    if peak > 0:
        result = result / peak * 0.95

    return result

if generate_audio and apply_effects and generated_tracks:
    # Utiliser le track "verse" pour les effets
    target = "verse" if "verse" in generated_tracks else list(generated_tracks.keys())[0]
    source_audio = generated_tracks[target]['audio']
    sr = generated_tracks[target]['sr']

    print(f"Application des effets sur : {target}")

    effects_results = {}

    # Reverb
    print(f"\n--- Reverb (decay=0.4, delay=50ms) ---")
    reverbed = apply_reverb(source_audio, sr, decay=0.4, delay_ms=50)
    effects_results["reverb"] = reverbed
    display_audio_array(reverbed, sr)

    # EQ : boost des basses
    print(f"\n--- EQ (bass boost) ---")
    eq_bass = apply_eq(source_audio, sr, bass_gain=1.5, mid_gain=1.0, treble_gain=0.8)
    effects_results["eq_bass_boost"] = eq_bass
    display_audio_array(eq_bass, sr)

    # EQ : voix claire (mid boost)
    print(f"\n--- EQ (mid boost - clarte) ---")
    eq_mid = apply_eq(source_audio, sr, bass_gain=0.8, mid_gain=1.3, treble_gain=1.1)
    effects_results["eq_clarity"] = eq_mid
    display_audio_array(eq_mid, sr)

    # Sauvegarder
    if save_audio_files:
        import soundfile as sf_lib
        for name, audio in effects_results.items():
            filepath = OUTPUT_DIR / f"effect_{name}.wav"
            sf_lib.write(str(filepath), audio, sr)
            print(f"Sauvegarde : {filepath.name}")

    print(f"\n{len(effects_results)} effets appliques")
else:
    print("Effets desactives ou pas de tracks disponibles")
    effects_results = {}


EFFETS AUDIO
Application des effets sur : verse

--- Reverb (decay=0.4, delay=50ms) ---


<audio embedded (audio/wav, 508204 bytes)>


--- EQ (bass boost) ---


<audio embedded (audio/wav, 508204 bytes)>


--- EQ (mid boost - clarte) ---


<audio embedded (audio/wav, 508204 bytes)>

Sauvegarde : effect_reverb.wav
Sauvegarde : effect_eq_bass_boost.wav
Sauvegarde : effect_eq_clarity.wav

3 effets appliques


### Interpretation : Effets audio

| Effet | Paramètre | Résultat typique | Usage |
|-------|-----------|------------------|-------|
| Reverb | decay=0.4, delay=50ms | Spatialisation legere | Ajouter de la profondeur |
| EQ bass boost | bass=1.5 | Sons plus chauds | Renforcer les basses |
| EQ mid boost | mid=1.3 | Clarte vocale | Faire ressortir les voix |

**Points cles** :
1. La reverb créé une impression d'espace mais peut reduire la clarte si excessive
2. L'EQ permet de corriger des problemes frequentiels ou de créer un style spécifique
3. Les effets sont cumulatifs : l'ordre d'application importe (typiquement EQ -> reverb)

> **Note technique** : Pour un traitement professionnel, utiliser des VST ou des DSP optimises (scipy reste adapte au prototypage rapide).

***

## Exercice : Chaîne d'effets audio personnalisable

**Duree estimee :** 20-25 minutes

### Objectif
Créer une fonction qui applique une chaîne d'effets audio (EQ, reverb, compression) dans un ordre configurable et exporte le résultat.

### Contexte
En production musicale, l'ordre d'application des effets est crucial. Appliquer une reverb avant un EQ donne un résultat différent d'un EQ avant une reverb. Cet exercice vous amene a construire un pipeline d'effes flexible.

### Instructions
1. Définir une liste d'effets a appliquer (EQ, reverb, gain)
2. Implementer chaque effet comme une fonction independante
3. Créer un pipeline qui applique les effets sequentiellement
4. Tester différentes configurations et comparer les résultats

> **Indices :**
- `# Étape 1` : Définir la liste d'effets : `[("eq", {"bass": 1.3}), ("reverb", {"decay": 0.3})]`
- `# Étape 2` : Utiliser `apply_eq()` et `apply_reverb()` du notebook
- `# Étape 3` : Appliquer chaque effet a la suite sur le signal modifie
- `# Indice` : Normaliser après chaque effet pour eviter l'accumulation de gain
- `# Indice` : Comparer EQ->reverb vs reverb->EQ pour entendre la différence

In [10]:
# TODO: Implementer la chaine d'effets personnalisable
def apply_effect_chain(audio: np.ndarray, sr: int,
                        effects_chain: list) -> np.ndarray:
    """
    Applique une chaine d'effets audio dans l'ordre specifie.
    
    Args:
        audio: Signal audio numpy array
        sr: Sample rate
        effects_chain: Liste de tuples (effect_name, params)
            ex: [("eq", {"bass_gain": 1.3}), ("reverb", {"decay": 0.3})]
    
    Returns:
        Audio traite par tous les effets
    """
    result = audio.copy()
    
    for effect_name, params in effects_chain:
        # Etape 1 : Dispatcher vers la bonne fonction d'effet
        # Indice: if effect_name == "eq": result = apply_eq(result, sr, **params)
        # Indice: if effect_name == "reverb": result = apply_reverb(result, sr, **params)
        pass  # TODO etudiant
        
        # Etape 2 : Normaliser apres chaque effet
        # Indice: peak = np.max(np.abs(result))
        # Indice: if peak > 0: result = result / peak * 0.95
        pass  # TODO etudiant
    
    return result

# Test avec differentes chaines
# audio_test = np.random.randn(44100).astype(np.float32) * 0.3  # bruit de test
# chain_1 = [("eq", {"bass_gain": 1.5}), ("reverb", {"decay": 0.4})]
# chain_2 = [("reverb", {"decay": 0.4}), ("eq", {"bass_gain": 1.5})]
# result_1 = apply_effect_chain(audio_test, 44100, chain_1)
# result_2 = apply_effect_chain(audio_test, 44100, chain_2)
# print(f"Chain EQ->Reverb : {len(result_1)} samples")
# print(f"Chain Reverb->EQ : {len(result_2)} samples")
print("Exercice a completer")

Exercice a completer


## Section 5 : Composition multi-sections

Une composition complete suit une structure musicale standard :

| Section | Duree typique | Caractère | Transition |
|---------|--------------|-----------|------------|
| Intro | 4-8s | Doux, progressif | Fade in |
| Couplet (Verse) | 8-16s | Energie moyenne | Crossfade |
| Refrain (Chorus) | 8-16s | Pleine energie | Crossfade |
| Outro | 4-8s | Decrescendo | Fade out |

Nous assemblons les sections generees en appliquant des transitions fluides.

### Exercice 3 : Variation melodique automatisee

Generez des variations d'une melodie en appliquant des transformations musicales (transposition, inversion, retrograde).

**Indice :** Representez la melodie comme une liste de notes MIDI et appliquez chaque transformation.

In [11]:
def generate_melody_variations(melody_notes, transformations=None):
    if transformations is None:
        transformations = ["transpose", "invert"]
    # Etape 1 : Appliquer la transposition (+/- demi-tons)
    # Etape 2 : Appliquer l'inversion (miroir vertical)
    # Etape 3 : Retourner un dictionnaire {nom_variation: notes}
    pass

result = None  # TODO etudiant
print("Exercice a completer")


Exercice a completer


### Preparation de l'assemblage

Avant d'assembler les sections en une composition complete, il faut définir la structure musicale. Les structures standard de la musique populaire comprennent typiquement :

- **Intro** : Etablissement de l'ambiance, instruments progressifs
- **Verse (Couplet)** : Narration, energie moyenne
- **Chorus (Refrain)** : Hook principal, energie maximale
- **Bridge** : Transition contrastante (optionnel)
- **Outro** : Conclusion, decrescendo

**Paramètres de transition** :
- **Crossfade** : Superposition progressive de deux sections (500ms)
- **Fade in/out** : Entree/sortie en douceur aux extremities
- **Normalisation** : Prevention du clipping (peak a -0.5 dBFS)

L'assemblage automatique ne tient pas compte du tempo ou de l'harmonie, mais créé des transitions fluides par des courbes de volume lineaires.

In [12]:
# Composition multi-sections avec transitions
print("COMPOSITION MULTI-SECTIONS")
print("=" * 50)

if generate_audio and generated_tracks:
    # Structure de la composition
    composition_structure = ["intro", "verse", "chorus", "verse", "chorus"]

    # Determiner le sample rate
    sr = list(generated_tracks.values())[0]['sr']

    # Parametres de transition
    fade_duration_ms = 500  # Duree du crossfade en ms
    fade_samples = int(sr * fade_duration_ms / 1000)

    print(f"Structure : {' -> '.join(composition_structure)}")
    print(f"Crossfade : {fade_duration_ms}ms ({fade_samples} samples)")

    # Construire la composition
    composition = np.array([], dtype=np.float32)

    for i, section_name in enumerate(composition_structure):
        if section_name not in generated_tracks:
            print(f"  Section '{section_name}' non disponible, saut")
            continue

        section_audio = generated_tracks[section_name]['audio'].copy()

        # Fade in pour la premiere section
        if i == 0:
            fade_in = np.linspace(0, 1, min(fade_samples, len(section_audio)))
            section_audio[:len(fade_in)] *= fade_in

        # Fade out pour la derniere section
        if i == len(composition_structure) - 1:
            fade_out = np.linspace(1, 0, min(fade_samples, len(section_audio)))
            section_audio[-len(fade_out):] *= fade_out

        # Crossfade avec la section precedente
        if i > 0 and len(composition) >= fade_samples:
            # Zone de crossfade
            xfade_len = min(fade_samples, len(section_audio))
            fade_out_curve = np.linspace(1, 0, xfade_len)
            fade_in_curve = np.linspace(0, 1, xfade_len)

            # Appliquer le crossfade
            composition[-xfade_len:] *= fade_out_curve
            section_audio[:xfade_len] *= fade_in_curve
            composition[-xfade_len:] += section_audio[:xfade_len]

            # Ajouter le reste de la section
            composition = np.concatenate([composition, section_audio[xfade_len:]])
        else:
            composition = np.concatenate([composition, section_audio])

        print(f"  [{i+1}] {section_name:8s} | {len(section_audio)/sr:.1f}s | Total : {len(composition)/sr:.1f}s")

    # Normalisation finale
    peak = np.max(np.abs(composition))
    if peak > 0:
        composition = composition / peak * 0.95

    print(f"\nComposition finale :")
    print(f"  Duree : {len(composition)/sr:.1f}s")
    print(f"  Sections : {len(composition_structure)}")
    print(f"  Sample rate : {sr}Hz")

    display_audio_array(composition, sr)

    # Sauvegarder
    if save_audio_files:
        import soundfile as sf
        filepath = OUTPUT_DIR / "composition_finale.wav"
        sf.write(str(filepath), composition, sr)
        print(f"\nSauvegarde : {filepath.name} ({filepath.stat().st_size/1024:.1f} KB)")
else:
    composition = np.array([])
    print("Composition desactivee (pas de tracks disponibles)")


COMPOSITION MULTI-SECTIONS
Structure : intro -> verse -> chorus -> verse -> chorus
Crossfade : 500ms (16000 samples)
  [1] intro    | 7.9s | Total : 7.9s
  [2] verse    | 7.9s | Total : 15.4s
  [3] chorus   | 7.9s | Total : 22.8s
  [4] verse    | 7.9s | Total : 30.3s
  [5] chorus   | 7.9s | Total : 37.7s

Composition finale :
  Duree : 37.7s
  Sections : 5
  Sample rate : 32000Hz


<audio embedded (audio/wav, 2412844 bytes)>


Sauvegarde : composition_finale.wav (2356.3 KB)


### Interpretation : Composition multi-sections

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| Crossfade | 500ms | Transition fluide entre sections |
| Fade in/out | 500ms | Entree/sortie progressive |
| Normalisation | -0.5 dBFS (0.95) | Marge de securite anti-clipping |

> **Note technique** : Pour une production plus avancee, utiliser des crossfades non-lineaires (courbes exponentielles) et des transitions musicalement coherentes (sur les temps forts).

## Section 6 : Export final avec metadonnees

L'export final ajoute des metadonnees ID3 au fichier audio pour une utilisation professionnelle.

In [13]:
# Export final avec metadonnees
print("EXPORT FINAL")
print("=" * 50)

if generate_audio and len(composition) > 0:
    sr = list(generated_tracks.values())[0]['sr']

    # Convertir numpy en AudioSegment pour l'export avec metadonnees
    # Normaliser en int16
    audio_int16 = (composition * 32767).astype(np.int16)
    audio_segment = AudioSegment(
        data=audio_int16.tobytes(),
        sample_width=2,
        frame_rate=sr,
        channels=1
    )

    # Export MP3 avec metadonnees
    export_path = OUTPUT_DIR / "production_finale.mp3"
    metadata = {
        "title": "Composition IA",
        "artist": "MusicGen + Demucs Pipeline",
        "album": "Cours Audio GenAI",
        "date": datetime.now().strftime("%Y"),
        "genre": "Electronic",
        "comment": f"Genere avec MusicGen ({musicgen_model}) et remixe avec Demucs ({demucs_model})"
    }

    tags = {f"-metadata {k}": v for k, v in metadata.items()}
    audio_segment.export(
        str(export_path),
        format="mp3",
        bitrate="256k",
        tags=metadata
    )

    print(f"Fichier exporte : {export_path.name}")
    print(f"Taille : {export_path.stat().st_size/1024:.1f} KB")
    print(f"Bitrate : 256 kbps")
    print(f"\nMetadonnees :")
    for key, value in metadata.items():
        print(f"  {key:10s} : {value}")

    print(f"\nEcoute de la production finale :")
    display_audio_file(export_path)
else:
    print("Export desactive (pas de composition disponible)")


EXPORT FINAL


Fichier exporte : production_finale.mp3
Taille : 1181.5 KB
Bitrate : 256 kbps

Metadonnees :
  title      : Composition IA
  artist     : MusicGen + Demucs Pipeline
  album      : Cours Audio GenAI
  date       : 2026
  genre      : Electronic
  comment    : Genere avec MusicGen (facebook/musicgen-medium) et remixe avec Demucs (htdemucs)

Ecoute de la production finale :


<audio embedded (audio/mpeg, 1209875 bytes)>

### Preparation de l'export

L'export est l'étape finale qui transforme la composition en un fichier partageable. Nous utilisons le format MP3 avec un bitrate de 256 kbps, qui offre un bon compromis entre qualite et taille de fichier.

**Metadonnees ID3** :
Les metadonnees permettent d'inclure des informations structurees dans le fichier audio :
- **title** : Nom de la piste
- **artist** : Auteur ou générateur
- **album** : Collection d'appartenance
- **genre** : Style musical
- **date** : Annee de creation
- **comment** : Description technique (modèles utilises, paramètres)

Ces informations sont lues par les lecteurs audio et permettent d'organiser et identifier les fichiers dans une bibliotheque musicale.

Le mode interactif permet de parametrer une composition personnalisee. Si le mode n'est pas `"interactive"`, cette section est ignoree et le notebook enchaine directement sur le bilan de session.

### Interpretation : Mode interactif

Le mode interactif permet d'explorer creativement les capacites de MusicGen en experimentant avec différentes descriptions textuelles.

**Avantages du mode interactif** :
- Experimentation rapide sans modifier le code
- Itération sur les prompts pour affiner le résultat
- Decouverte intuitive des capacites du modèle

**Contraintes** :
- Necessite une exécution en mode "interactive" (pas de batch MCP)
- Dependant de la disponibilite de MusicGen (transformers installe)
- La generation est toujours limitee a la duree specifiee dans les paramètres

> **Note** : En production, preferer des prompts pre-définis et testes pour garantir la coherence du résultat.

In [14]:
# Mode interactif - Composition personnalisee
if notebook_mode == "interactive" and not skip_widgets:
    print("MODE INTERACTIF - COMPOSITION PERSONNALISEE")
    print("=" * 50)
    print("\nDecrivez le style de musique que vous souhaitez generer :")
    print("(Laissez vide pour passer)")

    try:
        user_desc = input("\nDescription musicale : ")

        if user_desc.strip() and musicgen_available:
            print(f"\nGeneration en cours...")
            mg_model = MusicgenForConditionalGeneration.from_pretrained(
                musicgen_model, torch_dtype=torch.float16, device_map=device)
            mg_processor = AutoProcessor.from_pretrained(musicgen_model)
            mg_frame_rate = mg_model.config.audio_encoder.frame_rate

            inputs = mg_processor(text=[user_desc], padding=True, return_tensors="pt").to(device)
            audio_values = mg_model.generate(
                **inputs, do_sample=True, guidance_scale=3.0,
                max_new_tokens=int(generation_duration * mg_frame_rate)
            )
            audio = audio_values[0, 0].cpu().float().numpy()
            sr = mg_model.config.audio_encoder.sampling_rate

            print(f"Musique generee ({len(audio)/sr:.1f}s) :")
            display_audio_array(audio, sr)

            if save_audio_files:
                import soundfile as sf
                ts = datetime.now().strftime('%Y%m%d_%H%M%S')
                filepath = OUTPUT_DIR / f"custom_{ts}.wav"
                sf.write(str(filepath), audio, sr)
                print(f"Sauvegarde : {filepath.name}")

            del mg_model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        else:
            print("Mode interactif ignore")

    except (KeyboardInterrupt, EOFError):
        print("Mode interactif interrompu")
    except Exception as e:
        error_type = type(e).__name__
        if "StdinNotImplemented" in error_type or "input" in str(e).lower():
            print("Mode interactif non disponible (execution automatisee)")
        else:
            print(f"Erreur : {error_type} - {str(e)[:100]}")
else:
    print("Mode batch - Interface interactive desactivee")


Mode batch - Interface interactive desactivee


Le module de statistiques recapitule les résultats de la session : pieces generees, durees, formats exportes et ressources consommees par chaque étape de la composition.

In [15]:
# Statistiques de session
print("STATISTIQUES DE SESSION")
print("=" * 50)

print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode : {notebook_mode}")
print(f"MusicGen : {musicgen_model} | Demucs : {demucs_model}")

if generated_tracks:
    total_gen_time = sum(t['time'] for t in generated_tracks.values())
    total_duration = sum(t['duration'] for t in generated_tracks.values())
    print(f"Tracks generes : {len(generated_tracks)} ({total_duration:.1f}s total)")
    print(f"Temps de generation total : {total_gen_time:.1f}s")

if separated_stems:
    print(f"Stems separes : {len(separated_stems)}")

if remix_results:
    print(f"Versions remixees : {len(remix_results)}")

if len(composition) > 0:
    sr = list(generated_tracks.values())[0]['sr']
    print(f"Composition finale : {len(composition)/sr:.1f}s")

if save_audio_files:
    saved_files = list(OUTPUT_DIR.glob('*'))
    total_size = sum(f.stat().st_size for f in saved_files if f.is_file()) / 1024
    print(f"Fichiers sauvegardes : {len(saved_files)} ({total_size:.1f} KB)")

print(f"\nPROCHAINES ETAPES")
print(f"1. Synchroniser audio et video (04-4-Audio-Video-Sync)")
print(f"2. Explorer la serie Video (Video/01-Foundation)")

print(f"\nNotebook termine - {datetime.now().strftime('%H:%M:%S')}")


STATISTIQUES DE SESSION
Date : 2026-11-02 11:04:59
Mode : batch
MusicGen : facebook/musicgen-medium | Demucs : htdemucs
Tracks generes : 3 (23.8s total)
Temps de generation total : 169.3s
Stems separes : 4
Versions remixees : 4
Composition finale : 37.7s
Fichiers sauvegardes : 14 (9747.6 KB)

PROCHAINES ETAPES
1. Synchroniser audio et video (04-4-Audio-Video-Sync)
2. Explorer la serie Video (Video/01-Foundation)

Notebook termine - 11:04:59


***

## Exemple guide : Remix Musical Creatif

**Duree estimee :** 35-40 minutes

### Objectif
Créer un remix musical original en generant de la musique, separant les stems, et appliquant des effets creatifs.

### Instructions
1. Generer 3 sections musicales avec MusicGen (intro, couplet, refrain)
2. Separer les stems de la section la plus riche avec Demucs
3. Créer 3 versions remixees avec différentes configurations de mix
4. Appliquer des effets audio (reverb, EQ, delay) sur certains stems
5. Assembler une composition originale avec transitions

> **Indices :**
- Pour MusicGen: utilisez des prompts descriptifs (style, instrumentation, ambiance)
- Pour le remix: ajustez les gains de chaque stem (drums, bass, other, vocals)
- Pour les effets: utilisez les fonctions `apply_reverb()`, `apply_eq()` du notebook
- Pour l'assemblage: utilisez des crossfades de 500ms entre les sections

***

## Bilan de session

Les statistiques ci-dessous resument l'activite du pipeline de composition. Elles permettent d'evaluer l'efficacite de chaque étape et d'identifier les goulots d'etranglement (generation, separation, effets).

**Indicateurs cles** :
- Nombre de sections generees (intro, verse, chorus)
- Temps total de generation vs duree audio produite
- Nombre de stems separes (target = 4)
- Nombre de remixes crees
- Duree de la composition finale
- Espace disque utilise

Ces informations sont utiles pour optimiser le pipeline et planifier les ressources necessaires pour des projets plus ambitieux.

In [16]:
# Exercice: Generer 3 sections musicales
def generate_musical_sections() -> dict:
    """
    Genere 3 sections musicales avec MusicGen.
    
    Returns:
        Dict avec: {"intro": audio_data, "verse": audio_data, "chorus": audio_data}
    """
    # Exercice: Definir les prompts pour chaque section
    prompts = {
        "intro": "TODO: prompt pour intro (calme, progressive)",
        "verse": "TODO: prompt pour couplet (rythme, energie moyenne)",
        "chorus": "TODO: prompt pour refrain (energique, pleine puissance)"
    }
    
    # Indice: Generer chaque section avec MusicGen
    # Indice: mg_model.generate([prompt])
    # Indice: Convertir wav[0].cpu().numpy() en audio
    pass

# Exercice: Separer les stems d'une section
def separate_stems(audio_data: np.ndarray, sr: int) -> dict:
    """
    Separe les stems audio avec Demucs.
    
    Args:
        audio_data: Audio a separer
        sr: Sample rate
    
    Returns:
        Dict avec: {"drums": audio, "bass": audio, "other": audio, "vocals": audio}
    """
    # Indice: Sauvegarder l'audio temporairement
    # Indice: Executer Demucs via subprocess
    # Indice: subprocess.run([sys.executable, "-m", "demucs", "--name", "htdemucs", ...])
    # Indice: Charger les stems separes
    pass

# Exercice: Creer 3 remixes
def create_remixes(stems: dict, sr: int) -> dict:
    """
    Cree 3 versions remixees.
    
    Args:
        stems: Dict de stems separes
        sr: Sample rate
    
    Returns:
        Dict avec 3 remixes: {"original": mix, "instrumental": mix, "rhythmic": mix}
    """
    # Exercice: Definir les configurations de mix
    configs = {
        "original": {"drums": 1.0, "bass": 1.0, "other": 1.0, "vocals": 1.0},
        "instrumental": {"drums": 1.0, "bass": 1.0, "other": 1.2, "vocals": 0.0},
        # Exercice: Ajouter une 3e config (ex: "rhythmic")
    }
    
    # Indice: Mixer les stems selon chaque config
    # Indice: remix_stems() du notebook
    pass

# Exercice: Appliquer des effets
def apply_effects_to_remix(remix_audio: np.ndarray, sr: int) -> np.ndarray:
    """
    Applique des effets audio creatifs.
    
    Args:
        remix_audio: Audio a traiter
        sr: Sample rate
    
    Returns:
        Audio avec effets appliques
    """
    # Indice: Appliquer une reverb legere
    # Indice: apply_reverb(remix_audio, sr, decay=0.3)
    # Indice: Appliquer un EQ (boost basses ou aigus)
    # Indice: apply_eq(remix_audio, sr, bass_gain=1.2)
    pass

# Exercice: Assembler la composition finale
def assemble_composition(sections: dict, remixes: dict, sr: int) -> np.ndarray:
    """
    Assemble la composition finale avec transitions.
    
    Args:
        sections: Sections generees
        remixes: Remixes crees
        sr: Sample rate
    
    Returns:
        Composition complete
    """
    # Indice: Definir la structure (ex: intro -> verse -> chorus -> verse -> chorus)
    # Indice: Ajouter des crossfades entre les sections
    # Indice: utiliser pydub pour les transitions
    pass

# Exercice: Executer le pipeline complet
# sections = generate_musical_sections()
# stems = separate_stems(sections["chorus"], sr)
# remixes = create_remixes(stems, sr)
# composition = assemble_composition(sections, remixes, sr)

### Critères de succès
- [ ] 3 sections musicales generees avec des styles distincts
- [ ] Separation de stems fonctionnelle (4 stems)
- [ ] 3 remixes crees avec des configurations différentes
- [ ] Au moins 1 effet audio applique
- [ ] Composition finale assemblee avec transitions fluides

### Extension (optionnel)
- [ ] Generer une structure ABA (intro-verse-chorus-verse-chorus-outro)
- [ ] Créer un drop musical (transition percussive)
- [ ] Ajouter des samples sonores externes
- [ ] Exporter en format WAV haute qualite + metadonnees

***

## Synthese du module

Ce notebook a presente un pipeline complet de composition musicale assistee par IA, de la generation a l'export professionnel.

**Workflow complet** :
1. **Generation** : MusicGen transforme des descriptions textuelles en audio
2. **Separation** : Demucs isole les stems (drums, bass, other, vocals)
3. **Remixage** : Ajustement des gains de chaque stem
4. **Effets** : Reverb, EQ pour enrichir le son
5. **Assemblage** : Composition multi-sections avec transitions
6. **Export** : Fichier MP3 avec metadonnees

**Perspectives** :
- Integrer des LLMs pour generer des descriptions musicales plus variees
- Utiliser des modèles de style transfer pour appliquer des styles spécifiques
- Créer des compositions generatives infinies avec des boucles et variations
- Synchroniser l'audio avec des videos generees par IA (notebook suivant)

**Limitations actuelles** :
- MusicGen ne supporte pas la generation de stems separes directement
- La qualite audio (32kHz mono) est limitee pour la production professionnelle
- Les transitions automatiques ne tiennent pas compte du tempo ou de l'harmonie

**Prochaine étape** : Synchroniser audio et video dans le notebook 04-4-Audio-Video-Sync